In [ ]:
import rasterio
from rasterio.enums import Resampling as RioResampling
from rasterio.warp import reproject, Resampling as WarpResampling
from rasterio.mask import mask
import geopandas as gpd
import numpy as np
import pandas as pd



In [ ]:
# Load all US counties, reproject to match raster
counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip")
counties = counties.to_crs("EPSG:5070")

# Keep only CONUS
conus = counties[~counties["STATEFP"].isin(["02", "15", "60", "66", "69", "72", "78"])]

state_fips = {
    "01":"AL","04":"AZ","05":"AR","06":"CA","08":"CO","09":"CT",
    "10":"DE","11":"DC","12":"FL","13":"GA","16":"ID","17":"IL",
    "18":"IN","19":"IA","20":"KS","21":"KY","22":"LA","23":"ME",
    "24":"MD","25":"MA","26":"MI","27":"MN","28":"MS","29":"MO",
    "30":"MT","31":"NE","32":"NV","33":"NH","34":"NJ","35":"NM",
    "36":"NY","37":"NC","38":"ND","39":"OH","40":"OK","41":"OR",
    "42":"PA","44":"RI","45":"SC","46":"SD","47":"TN","48":"TX",
    "49":"UT","50":"VT","51":"VA","53":"WA","54":"WV","55":"WI","56":"WY"
}

pixel_area_ha = 250 * 250 / 10000  # 6.25 ha

# Find all surplus tif files
surplus_dir = Path("../../Surplus")
tif_files = sorted(surplus_dir.glob("Surplus_N_*.tif"))
print(f"Found {len(tif_files)} files: {[f.name for f in tif_files]}")

# Loop over years
all_years = []

for tif_path in tif_files:
    year = int(tif_path.stem.split("_")[-1])  # extract year from filename
    print(f"Processing {year}...")

    with rasterio.open(tif_path) as src:
        nodata = src.nodata  # use file's nodata if set, else None

    stats = zonal_stats(
        conus,
        str(tif_path),
        stats=["mean", "sum", "count"],
        nodata=nodata,
        geojson_out=False
    )

    df = pd.DataFrame(stats)
    df["year"]     = year
    df["GEOID"]    = conus["GEOID"].values
    df["STATEFP"]  = conus["STATEFP"].values
    df["COUNTYFP"] = conus["COUNTYFP"].values
    df["NAME"]     = conus["NAME"].values
    df["NAMELSAD"] = conus["NAMELSAD"].values
    df["STATE"]    = df["STATEFP"].map(state_fips)

    df["total_kg_N"] = df["sum"] * pixel_area_ha

    all_years.append(df)

# Combine into panel
panel = pd.concat(all_years, ignore_index=True)
panel = panel.rename(columns={"mean": "mean_surplus_kgha", "sum": "raw_sum", "count": "pixel_count"})
panel = panel[["year", "GEOID", "STATEFP", "STATE", "COUNTYFP", "NAME", "NAMELSAD",
               "mean_surplus_kgha", "raw_sum", "total_kg_N", "pixel_count"]]

panel.to_csv("nitrogen_surplus_by_county_panel.csv", index=False)
print(f"Done. Shape: {panel.shape}")
print(panel.head())

In [ ]:
df.head()

In [ ]:
print(panel.shape)               # expected: (n_counties × n_years, cols)
print(panel["year"].unique())    # confirm year range
print(panel.isnull().sum())      # any missing surplus values?
print(panel["mean_surplus_kgha"].describe())  # check for outliers, negatives

In [ ]:
import matplotlib.pyplot as plt
state_year = panel.groupby(["year", "STATE"])["mean_surplus_kgha"].mean().reset_index()

# Top 10 highest-surplus states
top_states = (panel.groupby("STATE")["mean_surplus_kgha"]
              .mean().sort_values(ascending=False).head(10).index)

for state in top_states:
    sub = state_year[state_year["STATE"] == state]
    plt.plot(sub["year"], sub["mean_surplus_kgha"], label=state)
plt.legend()

In [ ]:
dc = panel[panel["STATEFP"] == "11"]
print(dc[["year", "mean_surplus_kgha", "total_kg_N", "pixel_count"]].sort_values("year"))

In [ ]:
# Pick a single year
year_slice = panel[panel["year"] == 1950]
counties_geo = conus.merge(year_slice, on="GEOID")

counties_geo.plot(
    column="mean_surplus_kgha",
    cmap="RdYlGn_r",
    legend=True,
    figsize=(15, 8),
    missing_kwds={"color": "lightgrey"}
)
plt.title("County-Level N Surplus 2010 (kg/ha/yr)")

In [ ]:
urban_counties = ["11001",  # DC
                  "36061",  # New York (Manhattan)
                  "36047",  # Kings (Brooklyn)
                  "06075",  # San Francisco
                  "17031",  # Cook (Chicago)
                  "25025"]  # Suffolk (Boston)

check = (panel[panel["GEOID"].isin(urban_counties)]
         .groupby(["GEOID", "NAME", "STATE"])["mean_surplus_kgha"]
         .mean()
         .sort_values(ascending=False))
print(check)

In [ ]:
import xml.etree.ElementTree as ET

def get_iowa_cdl_url(year):
    url = f"https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLFile?year={year}&fips=19"
    r = requests.get(url)
    root = ET.fromstring(r.text)

    # Find returnURL regardless of namespace - iterate all elements
    for elem in root.iter():
        if elem.tag.endswith("returnURL"):
            return elem.text

    raise ValueError(f"No returnURL found in response: {r.text}")

# Test on 2017
file_url = get_iowa_cdl_url(2017)
print(file_url)

In [ ]:
years = range(1997, 2018)

for year in years:
    out_path = cdl_dir / f"CDL_{year}_19.tif"
    if out_path.exists():
        continue
    try:
        file_url = get_iowa_cdl_url(year)
        urllib.request.urlretrieve(file_url, out_path)
        print(f"{year}: saved")
    except Exception as e:
        print(f"{year}: failed - {e}")

In [ ]:
year = 2001
counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip")
iowa = counties[counties["STATEFP"]=="19"].to_crs("EPSG:5070")
iowa_geom = [iowa.union_all()]

cdl_legend = {1:"Corn", 5:"Soybeans", 24:"Winter Wheat", 36:"Alfalfa",
               176:"Grass/Pasture", 121:"Developed", 111:"Water", 141:"Forest"}

# --- Surplus: resample to 1km, then clip to Iowa ---
with rasterio.open(f"../../Surplus/Surplus_N_{year}.tif") as src:
    print("Surplus CRS:", src.crs)
    print("Surplus shape:", src.shape)

    scale = 250 / 1000
    new_h, new_w = int(src.height * scale), int(src.width * scale)
    new_transform = src.transform * src.transform.scale(src.width/new_w, src.height/new_h)

    surplus_1km = src.read(1, out_shape=(new_h, new_w), resampling=RioResampling.average)

    profile = src.profile.copy()
    profile.update(height=new_h, width=new_w, transform=new_transform)

    with rasterio.io.MemoryFile() as memfile:
        with memfile.open(**profile) as tmp:
            tmp.write(surplus_1km, 1)
        with memfile.open() as tmp:
            surplus_clip, clip_transform = mask(tmp, iowa_geom, crop=True, nodata=-999)
            surplus_clip = surplus_clip[0]

print("Surplus_clip shape:", surplus_clip.shape)
print("Surplus_clip dtype:", surplus_clip.dtype)
print("Valid pixels:", (surplus_clip != -999).sum())
print("Surplus range:", surplus_clip[surplus_clip != -999].min(), surplus_clip[surplus_clip != -999].max())

# --- CDL: check CRS before reprojecting ---
with rasterio.open(f"../../CDL/CDL_{year}_19.tif") as cdl_src:
    print("\nCDL CRS:", cdl_src.crs)
    print("CDL shape:", cdl_src.shape)
    print("CDL dtype:", cdl_src.dtypes[0])

    cdl_1km = np.empty(surplus_clip.shape, dtype=cdl_src.dtypes[0])
    reproject(
        source=rasterio.band(cdl_src, 1),
        destination=cdl_1km,
        src_transform=cdl_src.transform, src_crs=cdl_src.crs,
        dst_transform=clip_transform, dst_crs="EPSG:5070",
        resampling=WarpResampling.mode
    )

print("\nCDL_1km shape:", cdl_1km.shape)
print("CDL unique values:", np.unique(cdl_1km)[:20])  # first 20 unique classes

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14,6))

im0 = axes[0].imshow(np.where(surplus_clip==-999, np.nan, surplus_clip), cmap="RdYlGn_r")

axes[0].set_title(f"N Surplus {year} (1km)")

plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(cdl_1km, cmap="tab20")

axes[1].set_title(f"Dominant Crop {year} (1km)")

plt.colorbar(im1, ax=axes[1])

plt.show()